In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from IPython.display import display, HTML, SVG
display(HTML('<style>.container { width:90% !important; }</style>'))
import requests
from bs4 import BeautifulSoup
import numpy as np
from datetime import datetime
from random import randint
import pandas as pd
pd.set_option("display.date_dayfirst", True)
# pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
import os
from flask import Flask, render_template, request, url_for, redirect
from flask_sqlalchemy import SQLAlchemy
from tqdm.notebook import tqdm, trange
import time
import re
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="myBestSecretAgent")
import sqlite3

In [ ]:
from application.config import basedir
from dotenv import load_dotenv

load_dotenv(os.path.join(basedir, '.env'))

In [ ]:
basedir

In [ ]:
engine = create_engine(f"postgresql://{os.environ.get('USERNAME')}:{os.environ.get('PASSWORD')}@{os.environ.get('HOST')}:5432/{os.environ.get('DATABASE')}")

In [ ]:
import os

print('DATABASE_URL: ', os.environ.get('DATABASE_URL'))
print('SQLALCHEMY_DATABASE_URI: ', os.environ.get('SQLALCHEMY_DATABASE_URI'))
print('SECRET_KEY: ', os.environ.get('SECRET_KEY'))

In [ ]:
from application.config import config
server = Flask(__name__)
server.config.from_object(config.get('default'))
for k, v in server.config.items():
    print(k, v)

In [ ]:
config.get('default')

In [ ]:
del os
import os

In [ ]:
os.environ

# CLEANING DATA

#to change output windows' heights  
css = ".output_wrapper { min-height: none; }"  
HTML('<style>{}</style>'.format(css))

In [ ]:
fp = "app/data/xls_x/carnet_de_bord.xlsx"

## PDS

In [ ]:
converters={
    'nom': str,
    'prenom': str,
    'spe': str,
    'uga': str,
    'etablissement': str,
    'adresse': str,
    'cp': int,
    'ville': str,
    'tel': str,
    'pot': int,
    'pvm': int,
    'nv2022': int,
    'commentaire': str,
    'lun_mat': str,
    'lun_am': str,
    'mar_mat': str,
    'mar_am': str,
    'mer_mat': str,
    'mer_am': str,
    'jeu_mat': str,
    'jeu_am': str,
    'ven_mat': str,
    'ven_am': str,
    'ciblage': int
}

pds = pd.read_excel(fp, sheet_name="pds", skiprows=[2078], converters=converters)
pds.columns = [
    'nom', 'prenom', 'spe', 'uga', 'etablissement', 'adresse', 'cp',
    'ville', 'tel', 'pot', 'pvm', 'nv2022', 'mode', 'commentaire', 'lun_mat', 'lun_am', 'mar_mat', 'mar_am', 'mer_mat', 'mer_am',
    'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ddv', 'ciblage'
]

pds["ddv"] = pd.to_datetime(pds["ddv"], dayfirst=True, format="%d/%m/%Y %H:%M")
pds["ddv"] = [int(row["ddv"].timestamp()) if pd.isnull(row["ddv"]) == 0 else '' for i, row in pds.iterrows()]

regex=re.compile('[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}')

for i, row in pds.iterrows():
    s=re.search(regex, str(row['mode']))
    if s:
        pds.loc[i, 'mode'] = int(pd.to_datetime(row['mode'], format="%Y-%m-%d %H:%M:%S").timestamp())

for col in ["pot", "pvm", "nv2022", "ciblage"]:
    pds[col] = pds[col].fillna(0)

for col in ['nom', 'prenom', 'spe', 'uga', 'etablissement', 'adresse', 'cp',
       'ville', 'tel', 'mode', 'commentaire',
       'lun_mat', 'lun_am', 'mar_mat', 'mar_am', 'mer_mat', 'mer_am',
       'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ciblage']:
    pds[col] = pds[col].fillna('')
    pds[col] = np.where(pds[col] == 'nan', '', pds[col])

pds["tel"] = ["0" + str(row["tel"]) if row["tel"] != "" else '' for i, row in pds.iterrows() ]

pds["tel"] = [str(row["tel"][:2]+" "+row["tel"][2:4]+" "+row["tel"][4:6]+" "+row["tel"][6:8]+" "+row["tel"][8:10]) if row["tel"] != "" else '' for i, row in pds.iterrows() ]

pds["id"] = pds.index + 1

pds = pds[
    [
        'id', 'nom', 'prenom', 'spe', 'ddv',  'pot', 'pvm', 'nv2022', 'ciblage', 
        'uga', 'etablissement', 'adresse', 'cp','ville', 'tel', 
        'mode', 'commentaire', 'lun_mat', 'lun_am', 'mar_mat', 'mar_am', 'mer_mat', 'mer_am','jeu_mat', 'jeu_am', 'ven_mat', 'ven_am']
]

identities = pds[["id", "nom", "prenom", "spe", "pot", "pvm", "nv2022", "ciblage"]].copy()

def get_identity(docId):
    return pd.DataFrame(identities[identities["id"] == docId])

adresses = pds[["uga", "etablissement", "adresse", "cp", "ville", "tel"]].copy()
adresses.drop_duplicates(subset=["uga", "etablissement", "adresse", "cp", "ville"], inplace=True)
adresses.reset_index(drop=True, inplace=True)
adresses["id"] = adresses.index + 1
adresses = adresses[["id", "uga", "etablissement", "adresse", "cp", "ville", "tel"]]

## ADRESSES

### IF NOT DONE

In [ ]:
adresses["lat"] = pd.Series(dtype=float)
adresses["lon"] = pd.Series(dtype=float)

errors = []
for i, row in tqdm(adresses.iterrows(), total=len(adresses)):
    adr = row["adresse"] + " " + str(row["cp"]) + " " +row["ville"]
    try:
        location = geolocator.geocode(adr)
        if location is not None:
            adresses.loc[i, "lat"] = location.latitude
            adresses.loc[i, "lon"] = location.longitude
        else:
            errors.append(i)
            print(f"problem at index {i} with adress {adr}")
    except:
        errors.append(i)
        print(f"problem at index {i} with adress {adr}")

In [ ]:
for i in errors:
    row = pds.loc[i]
    adr = row["adresse"] + " " + str(row["cp"]) + " " +row["ville"]
    try:
        location = geolocator.geocode(adr)
        if location is not None:
            adresses.loc[i, "lat"] = location.latitude
            adresses.loc[i, "lon"] = location.longitude
            errors.remove(i)
        else:
            errors.append(i)
            print(f"problem at index {i} with adress {adr}")
    except:
        errors.append(i)
        print(f"problem at index {i} with adress {adr}")

In [ ]:
errors

In [ ]:
def get_adress_id(docId):
    row = pds[pds["id"]==docId]
    return adresses["id"][
        (adresses["etablissement"]==row["etablissement"].values[0]) 
        & 
        (adresses["uga"]==row["uga"].values[0]) 
        & 
        (adresses["adresse"]==row["adresse"].values[0]) 
        & 
        (adresses["cp"]==row["cp"].values[0]) 
        & 
        (adresses["ville"]==row["ville"].values[0])
    ].values[0]

In [ ]:
home = geolocator.geocode("35 RUE MARCEL BONTEMPS 92100 BOULOGNE BILLANCOURT")

## CONNECTIONS

In [ ]:
connections = pd.DataFrame(dtype=int, index=identities.index)
connections["doc_id"] = identities["id"]

In [ ]:
connections["adress_id"] = [get_adress_id(docId) for docId in connections["doc_id"]]

In [ ]:
pds["adress_id"] = [get_adress_id(row["id"]) for i, row in pds.iterrows()]

In [ ]:
def get_adresses(docId):
    
    result = pd.DataFrame(dtype=object, columns=["uga", "etablissement", "adresse", "cp", "ville", "tel", "id"])
    arr = connections["adress_id"][connections["doc_id"] == docId].values
    
    for adr_id in arr:
        result = pd.concat([result, pd.DataFrame(adresses[adresses["id"]==adr_id])], ignore_index=True)

    return result[["id", "uga", "etablissement", "adresse", "cp", "ville", "tel"]]

In [ ]:
get_identity(1069)
get_adresses(1069)

## CARNETS DE BORD

In [ ]:
cdb_cols = ["id", "adress_id", "mode", "commentaire", 'lun_mat', 'lun_am', 'mar_mat', 'mar_am','mer_mat', 'mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ddv']
cdbs = pds[cdb_cols].copy()
cdbs.columns = ["doc_id", "adress_id", "mode", "commentaire", 'lun_mat', 'lun_am', 'mar_mat', 'mar_am','mer_mat', 'mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ddv']

In [ ]:
cdbs["id"] = cdbs.index + 1
cdbs["id"] = cdbs["id"].astype(int)
cdbs = cdbs[["id", "doc_id", "adress_id", "mode", "commentaire", 'lun_mat', 'lun_am', 'mar_mat', 'mar_am','mer_mat', 'mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ddv']]

In [ ]:
connections["cdb_id"] = [cdbs["id"][(cdbs["doc_id"]==row["doc_id"])&(cdbs["adress_id"]==row["adress_id"])].values[0] for i, row in connections.iterrows()]

In [ ]:
connections['id'] = connections.index + 1

In [ ]:
connections = connections[["id", "doc_id", "adress_id", "cdb_id"]]

In [ ]:
def get_cdbs(docId):
    
    adrs = get_adresses(docId)    
    result = pd.DataFrame(dtype=object, columns=["id", "doc_id", "adress_id", "mode", "commentaire", 'lun_mat', 'lun_am', 'mar_mat', 'mar_am','mer_mat', 'mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am', 'ddv'])
    
    for i in range(adrs.shape[0]):
        cdb = cdbs[(cdbs["doc_id"]==docId)&(cdbs["adress_id"]==adrs.loc[i, "id"])]
        result = pd.concat([result, cdb], ignore_index=True)
    
    return result

In [ ]:
get_cdbs(1069)

In [ ]:
cdbs.drop(["doc_id", "adress_id"], axis=1, inplace=True)

In [ ]:
identities.columns
adresses.columns
cdbs.columns
connections.columns

# SQLITE DATABASE

In [ ]:
create_id  ="""
CREATE TABLE identities (
    id INTEGER PRIMARY KEY NOT NULL,
    nom TEXT NOT NULL,
    prenom TEXT,
    spe TEXT,
    pot INTEGER,
    pvm INTEGER,
    nv2022 INTEGER,
    ciblage INTEGER
    );
"""

In [ ]:
create_ad = """
CREATE TABLE adresses (
    id INTEGER PRIMARY KEY NOT NULL,
    uga TEXT NOT NULL,
    etablissement TEXT,
    adresse TEXT NOT NULL,
    cp TEXT NOT NULL,
    ville TEXT NOT NULL,
    tel TEXT,
    lat REAL DEFAULT 0,
    lon REAL DEFAULT 0
    );
"""

In [ ]:
create_cd  = """
CREATE TABLE cdbs (
    id INTEGER PRIMARY KEY NOT NULL,
    mode TEXT,
    commentaire TEXT,
    lun_mat TEXT,
    lun_am TEXT,
    mar_mat TEXT,
    mar_am TEXT,
    mer_mat TEXT,
    mer_am TEXT,
    jeu_mat TEXT,
    jeu_am TEXT,
    ven_mat TEXT,
    ven_am TEXT,
    ddv INTEGER
    );
"""

In [ ]:
create_map = """
CREATE TABLE connections (
    id INTEGER PRIMARY KEY NOT NULL,
    doc_id INTEGER NOT NULL,
    adress_id INTEGER NOT NULL,
    cdb_id INTEGER NOT NULL,
    FOREIGN KEY(doc_id) REFERENCES identities(id),
    FOREIGN KEY(adress_id) REFERENCES adresses(id),
    FOREIGN KEY(cdb_id) REFERENCES cdbs(id)
    );
"""

In [ ]:
create_users = """
CREATE TABLE users (
    id INTEGER PRIMARY KEY NOT NULL,
    username TEXT,
    email TEXT NOT NULL,
    password_hash TEXT NOT NULL,
    last_seen INTEGER DEFAULT CURRENT_TIMESTAMP,
    token TEXT,
    token_expiration INTEGER ,
    about_me TEXT
    );
"""

In [ ]:
from dotenv import load_dotenv
import psycopg2

basedir = os.getcwd()
load_dotenv(os.path.join(basedir, '.env'))

In [ ]:
conn = psycopg2.connect(
        host=os.environ.get('HOST'),
        database=os.environ.get('DATABASE'),
        user=os.environ.get('USERNAME'),
        password=os.environ.get('PASSWORD')
)

cursor = conn.cursor()

In [ ]:
try:
    for query in [create_id, create_cd, create_ad, create_map]:
        cursor.execute(query)
except psycopg2.Error as error:
    print('Error occurred - ', error)

In [ ]:
conn.commit()
cursor.close()
conn.close()

In [ ]:
identities=pd.read_csv('application/data/csv/identities.csv', sep=";")
adresses=pd.read_csv('application/data/csv/adresses.csv', sep=";")
cdbs=pd.read_csv('application/data/csv/cdbs.csv', sep=";")
connections=pd.read_csv('application/data/csv/connections.csv', sep=";")

In [ ]:
from sqlalchemy import create_engine
engine = create_engine(f"postgresql://{os.environ.get('USERNAME')}:{os.environ.get('PASSWORD')}@{os.environ.get('HOST')}:5432/{os.environ.get('DATABASE')}")

identities.to_sql('identities', engine, if_exists='append', index=False)
adresses.to_sql('adresses', engine, if_exists='append', index=False)
cdbs.to_sql('cdbs', engine, if_exists='append', index=False)
connections.to_sql('connections', engine, if_exists='append', index=False)

In [ ]:
identities.to_csv('app/data/csv/identities.csv', sep=";", index=False)
adresses.to_csv('app/data/csv/adresses.csv', sep=";", index=False)
cdbs.to_csv('app/data/csv/cdbs.csv', sep=";", index=False)
connections.to_csv('app/data/csv/connections.csv', sep=";", index=False)

In [ ]:
from dash import html

In [ ]:
def discrete_background_color_bins(df, n_bins=5, columns='all', colorscale="Blues"):
    import colorlover
    bounds = [i * (1.0 / n_bins) for i in range(n_bins + 1)]
    if columns == 'all':
        if 'id' in df:
            df_numeric_columns = df.select_dtypes('int64').drop(['id'], axis=1)
        else:
            df_numeric_columns = df.select_dtypes('int64')
    else:
        df_numeric_columns = df[columns]
    df_max = df_numeric_columns.max().max()
    df_min = df_numeric_columns.min().min()
    ranges = [
        ((df_max - df_min) * i) + df_min
        for i in bounds
    ]
    styles = []
    legend = []
    for i in range(1, len(bounds)):
        min_bound = ranges[i - 1]
        max_bound = ranges[i]
        backgroundColor = colorlover.scales[str(n_bins)]['seq'][colorscale][i - 1]
        color = 'white' if i > len(bounds) / 2. else 'inherit'

        for column in df_numeric_columns:
            styles.append({
                'if': {
                    'filter_query': (
                        '{{{column}}} >= {min_bound}' +
                        (' && {{{column}}} < {max_bound}' if (i < len(bounds) - 1) else '')
                    ).format(column=column, min_bound=min_bound, max_bound=max_bound),
                    'column_id': column
                },
                'backgroundColor': backgroundColor,
                'color': color
            })
        legend.append(
            html.Div(style={'display': 'inline-block', 'width': '30px'}, children=[
                html.Div(
                    style={
                        'backgroundColor': backgroundColor,
                        'borderLeft': '1px rgb(50, 50, 50) solid',
                        'height': '10px'
                    }
                ),
                html.Small(int(min_bound), style={
                    'paddingLeft': '2px',
                    'color': 'black',
                    'textOrientation': 'upright',
                    'fontSize': 8}
                           )
            ])
        )

    return (styles, html.Div(legend, style={'padding': '5px 0 5px 0', 'color': 'white'}))

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from application.dashboards.biocodex.models import Identity, Adress, Cdb, Connections
import pandas as pd
from dash import html
import dash_bootstrap_components as dbc
from flask import url_for

engine = create_engine(f"postgresql://{os.environ.get('USERNAME')}:{os.environ.get('PASSWORD')}@{os.environ.get('HOST')}:5432/{os.environ.get('DATABASE')}")

def join_id_adr():

    with Session(engine) as session:
        id_adr = pd.read_sql_query(
            sql=session.query(
                Connections.doc_id, Identity.nom, Identity.prenom, Identity.spe, Identity.pot, Identity.pvm,
                Identity.nv2022, Identity.ciblage, Adress.etablissement, Adress.uga, Adress.adresse, Adress.cp, Adress.ville, Adress.tel,
                Adress.lat, Adress.lon
            ).join(Identity).join(Adress).statement,
            con=engine
        )
        session.close()
        id_adr.columns = [
            'id', 'nom', 'prenom', 'spe', 'pot', 'pvm', 'nv2022', 'ciblage', 'etablissement',
            'uga', 'adresse', 'cp', 'ville', 'tel', 'lat', 'lon'
        ]
    return id_adr


def join_id_cdb():

    with Session(engine) as session:
        id_cdb = pd.read_sql_query(
            sql=session.query(
                Connections.doc_id, Identity.nom, Identity.prenom, Identity.spe, Identity.pot, Identity.pvm,
                Identity.nv2022,
                Cdb.lun_mat, Cdb.lun_am, Cdb.mar_mat, Cdb.mar_am, Cdb.mer_mat, Cdb.mer_am, Cdb.jeu_mat, Cdb.jeu_am,
                Cdb.ven_mat, Cdb.ven_am, Cdb.ddv
            ).join(Identity).join(Cdb).statement,
            con=engine
        )
        session.close()

    return id_cdb


def join_id_adr_cdb():

    with Session(engine) as session:
        id_adr_cdb = pd.read_sql_query(
            sql=session.query(
                Connections.doc_id, Identity.nom, Identity.prenom, Identity.spe, Identity.pot, Identity.pvm,
                Identity.nv2022, Identity.ciblage,
                Adress.etablissement, Adress.uga, Adress.adresse, Adress.cp, Adress.ville, Adress.tel, Adress.lat, Adress.lon,
                Cdb.mode, Cdb.commentaire, Cdb.lun_mat, Cdb.lun_am, Cdb.mar_mat, Cdb.mar_am, Cdb.mer_mat, Cdb.mer_am, Cdb.jeu_mat, Cdb.jeu_am,
                Cdb.ven_mat, Cdb.ven_am, Cdb.ddv
            ).join(Identity).join(Adress).join(Cdb).statement,
            con=engine
        )
        session.close()

    return id_adr_cdb

In [ ]:
df = join_id_adr_cdb()
df.sample()

## FROM DATABASE

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

In [ ]:
from app.dashboards.biocodex.models import Identity, Connections, Adress, Cdb

In [ ]:
from app.dashboards.biocodex.functions import join_id_adr_cdb

In [ ]:
df =  join_id_adr_cdb()
df.shape

In [ ]:
df.sample(10)

In [ ]:
ciblage = df[df["ciblage"]!=0].copy()

In [ ]:
ciblage.shape

In [ ]:
piv_cib=ciblage.pivot_table(values="nom", index="spe", columns="uga", aggfunc='count', margins=True).fillna(0).astype(int)

In [ ]:
piv_cib

In [ ]:
piv_cib/piv_pds

# PHARMACIES

In [ ]:
pharma = pd.read_csv("/home/julien/website/app/data/csv/pharmacies.csv", sep=";")

pharma["lat"] = pd.Series(dtype=float)
pharma["lon"] = pd.Series(dtype=float)

pharma.columns

from tqdm.notebook import tqdm, trange

from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="mySecretAgent")

for i, row in tqdm(pharma.iterrows(), total=len(pharma)):
    adr = row["adresse"] + " " + str(row["cp"]) + " " +row["ville"]
    location = geolocator.geocode(adr)
    if location is not None:
        pharma.loc[i, "lat"] = location.latitude
        pharma.loc[i, "lon"] = location.longitude
    else:
        print(i, adr)

pharma.head()

# FOLIUM

In [ ]:
ciblage.sample()
pharma.sample()

In [ ]:
import numpy as np
import pandas as pd
import folium

color : str, default 'blue'
    The color of the marker. You can use:

        ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
         'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
         'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen',
         'gray', 'black', 'lightgray']

icon_color : str, default 'white'
    The color of the drawing on the marker. You can use colors above,
    or an html color code.
      
icon : str, default 'info-sign'
    The name of the marker sign.
    See Font-Awesome website to choose yours.
    Warning : depending on the icon you choose you may need to adapt
    the `prefix` as well.  
    
angle : int, default 0
    The icon will be rotated by this amount of degrees.  
    
prefix : str, default 'glyphicon'
    The prefix states the source of the icon. 'fa' for font-awesome or
    'glyphicon' for bootstrap 3.

In [ ]:
data = df.to_dict('records')

In [ ]:
dff = pd.DataFrame(data)
dff.head()

In [ ]:
map = folium.Map(
    location=[
        ciblage["lat"].mean(), 
        ciblage["lon"].mean()
    ],
    zoom_start=12,
    control_scale=True
)

# Loop through each row in the dataframe
for i,row in pharma.iterrows():
    #Setup the content of the popup
    iframe = folium.IFrame('Nom:\t' + str(row["nom"]) + '\nUGA:\t ' + str(row["uga"]))
    
    #Initialise the popup using the iframe
    popup = folium.Popup(iframe, min_width=300, max_width=300)
    
    #Add each row to the map
    folium.Marker(
        location=[row['lat'],row['lon']],
        popup=popup,
        icon=folium.Icon(color="green", prefix="fa", icon="prescription-bottle-alt")
    ).add_to(map)

In [ ]:
icon=folium.Icon(color="green", prefix="fa", icon="prescription-bottle-alt")

icon.to_json()

In [ ]:
map

In [ ]:
spe_to_color={
    "GY": "pink",
    "MG-GY": "pink",
    "SF": "pink",
    "MG": "gray",
    "PE": "lightblue",
    "GE": "orange",
    "PE_PSY": "darkblue",
    "NE": "darkpurple"
}

In [ ]:
ciblage["spe"].unique()

In [ ]:
# Loop through each row in the dataframe
for i,row in ciblage[ciblage["uga"]=="75GRE"].iterrows():

    spe = row["spe"]
    #Setup the content of the popup
    iframe = folium.IFrame(f'{row["nom"]} {row["prenom"]}\n{row["spe"]}')
    
    #Initialise the popup using the iframe
    popup = folium.Popup(iframe, min_width=300, max_width=300)
    
    #Add each row to the map
    folium.Marker(
        location=[row['lat'],row['lon']],
        popup=popup,
        icon=folium.Icon(color=spe_to_color[spe], prefix="fa", icon="user-md")
    ).add_to(map)

In [ ]:
map

In [ ]:
with open("test.html", "w") as f:
    f.write(map._repr_html_())

# DASH

In [ ]:
from app.dashboards.biocodex.models import Identity, Adress, Cdb, Connections

In [ ]:
from dash import Dash, html, Input, Output, callback, State
from dash.exceptions import PreventUpdate
import dash_bootstrap_components as dbc

from dash import Dash, html, dcc, dash_table

In [ ]:
from ipyleaflet import AwesomeIcon, Marker, Map

In [ ]:
map = Map(center=[ciblage["lat"].mean(), ciblage["lon"].mean()], zoom=12)

In [ ]:
from ipyleaflet import Map, basemaps, basemap_to_tiles

m = Map(
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    center=(48.204793, 350.121558),
    zoom=3
    )
m

In [ ]:
from dash_iconify import DashIconify

In [ ]:
children = [dl.TileLayer()]

In [ ]:
icon=DashIconify(
            
                icon="ion:logo-github",
                width=30,
                height=30,
                rotate=1,
                flip="horizontal",
            )

In [ ]:
from cairosvg import svg2png

In [ ]:
svg_code = """
    <svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" aria-hidden="true" role="img" loading_state="[object Object]" class="iconify iconify--ion" width="30" height="30" preserveAspectRatio="xMidYMid meet" viewBox="0 0 512 512"><g transform="rotate(90 256 256) translate(512 0) scale(-1 1)"><path fill="currentColor" d="M256 32C132.3 32 32 134.9 32 261.7c0 101.5 64.2 187.5 153.2 217.9a17.56 17.56 0 0 0 3.8.4c8.3 0 11.5-6.1 11.5-11.4c0-5.5-.2-19.9-.3-39.1a102.4 102.4 0 0 1-22.6 2.7c-43.1 0-52.9-33.5-52.9-33.5c-10.2-26.5-24.9-33.6-24.9-33.6c-19.5-13.7-.1-14.1 1.4-14.1h.1c22.5 2 34.3 23.8 34.3 23.8c11.2 19.6 26.2 25.1 39.6 25.1a63 63 0 0 0 25.6-6c2-14.8 7.8-24.9 14.2-30.7c-49.7-5.8-102-25.5-102-113.5c0-25.1 8.7-45.6 23-61.6c-2.3-5.8-10-29.2 2.2-60.8a18.64 18.64 0 0 1 5-.5c8.1 0 26.4 3.1 56.6 24.1a208.21 208.21 0 0 1 112.2 0c30.2-21 48.5-24.1 56.6-24.1a18.64 18.64 0 0 1 5 .5c12.2 31.6 4.5 55 2.2 60.8c14.3 16.1 23 36.6 23 61.6c0 88.2-52.4 107.6-102.3 113.3c8 7.1 15.2 21.1 15.2 42.5c0 30.7-.3 55.5-.3 63c0 5.4 3.1 11.5 11.4 11.5a19.35 19.35 0 0 0 4-.4C415.9 449.2 480 363.1 480 261.7C480 134.9 379.7 32 256 32Z"></path></g></svg>
"""

svg2png(bytestring=svg_code,write_to='output.png')

In [ ]:
custom_icon = dict(
    iconUrl='output.png',
    # shadowUrl='https://leafletjs.com/examples/custom-icons/leaf-shadow.png',
    iconSize=[38, 95],
    shadowSize=[50, 64],
    iconAnchor=[22, 94],
    shadowAnchor=[4, 62],
    popupAnchor=[-3, -76]
)

In [ ]:
for i,row in pharma.iterrows():
    #Setup the content of the popup
    #iframe = folium.IFrame('Nom:\t' + str(row["nom"]) + '\nUGA:\t ' + str(row["uga"]))
    
    #Initialise the popup using the iframe
    #popup = folium.Popup(iframe, min_width=300, max_width=300)
    
    #Add each row to the map
    children.append(
        dl.Marker(
            icon=custom_icon,
            position=[row['lat'],row['lon']],
        )
    )

In [ ]:
#BOOTSTRAP LAYOUT EXAMPLE

app = Dash(external_stylesheets=[dbc.themes.CERULEAN, {"href":"app/static/css/custom.css", "type": "tex/css"}])

app.layout = \
dbc.Container\
([
    dl.Map(children, center=[ciblage["lat"].mean(), ciblage["lon"].mean()], zoom=12, style={'height': '50vh'}),
    html.Br(),
    dbc.Row([
    dbc.Col([dbc.Button("row 1 col 1",style={"width":"100%"})],width=3),
    dbc.Col([dbc.Button("row 1 col 2", style={"width": "100%"})],width=3),
    dbc.Col([dbc.Button("row 1 col 3",style={"width":"100%"})],width=3),
    dbc.Col([dbc.Button("row 1 col 4",style={"width":"100%"})],width=3),
    ]),
    html.Br(),
    dbc.Row([
    dbc.Col([dbc.Button("row 2 col 1",style={"width":"100%"})],width=3),
    dbc.Col([dbc.Button("row 2 col 2", style={"width": "100%"})],width=3),
    dbc.Col([dbc.Button("row 2 col 3",style={"width":"100%"})],width=6),
    ]),
    html.Br(),
    dbc.Row([
    dbc.Col([dbc.Button("row 3 col 1",style={"width":"100%"})],width=9),
    dbc.Col([dbc.Button("row 3 col 2", style={"width": "100%"})],width=3),
    ])
])

if __name__ == "__main__":
    app.run_server(debug=False, port=8050, host='0.0.0.0')

In [ ]:
def discrete_background_color_bins(df, n_bins=5, columns='all', colorscale="Blues"):
    import colorlover
    bounds = [i * (1.0 / n_bins) for i in range(n_bins + 1)]
    if columns == 'all':
        if 'id' in df:
            df_numeric_columns = df.select_dtypes('number').drop(['id'], axis=1)
        else:
            df_numeric_columns = df.select_dtypes('number')
    else:
        df_numeric_columns = df[columns]
    df_max = df_numeric_columns.max().max()
    df_min = df_numeric_columns.min().min()
    ranges = [
        ((df_max - df_min) * i) + df_min
        for i in bounds
    ]
    styles = []
    legend = []
    for i in range(1, len(bounds)):
        min_bound = ranges[i - 1]
        max_bound = ranges[i]
        backgroundColor = colorlover.scales[str(n_bins)]['seq'][colorscale][i - 1]
        color = 'white' if i > len(bounds) / 2. else 'black'

        for column in df_numeric_columns:
            styles.append({
                'if': {
                    'filter_query': (
                        '{{{column}}} >= {min_bound}' +
                        (' && {{{column}}} < {max_bound}' if (i < len(bounds) - 1) else '')
                    ).format(column=column, min_bound=min_bound, max_bound=max_bound),
                    'column_id': column
                },
                'backgroundColor': backgroundColor,
                'color': color
            })
        legend.append(
            html.Div(style={'display': 'inline-block', 'width': '60px'}, children=[
                html.Div(
                    style={
                        'backgroundColor': backgroundColor,
 org/en/14/orm/internals.html                       'borderLeft': '1px rgb(50, 50, 50) solid',
                        'height': '10px'
                    }
                ),
                html.Small(round(min_bound, 2), style={'paddingLeft': '2px'})
            ])
        )

    return (styles, html.Div(legend, style={'padding': '5px 0 5px 0'}))

In [ ]:
id_adr = join_id_adr()

In [ ]:
(styles1, legend1) = discrete_background_color_bins(id_adr, n_bins=9, columns=["pvm"])
(styles2, legend2) = discrete_background_color_bins(id_adr, n_bins=9, columns=["pot"], colorscale="YlGn")
styles = styles1 + styles2

In [ ]:
spe_options = [{"label": val, "value": val} for val in [id_adr["spe"].unique().tolist()[i] for i in [6, 3, -1, 1 ,-2 ,5 ,4 ,0 ,2]]]
uga_options = [{"label": val, "value": val} for val in id_adr["uga"].unique().tolist()]
cp_options = [{"label": val, "value": val} for val in id_adr["cp"].unique().tolist()]
ville_options = [{"label": val, "value": val} for val in id_adr["ville"].unique().tolist()]

In [ ]:
CSS = """
.dash-spreadsheet.dash-freeze-top, .dash-spreadsheet.dash-virtualized { max-height: inherit !important; }    
.dash-table-container {max-height: calc(100vh - 125px);}
"""
HTML('<style>{}</style>'.format(CSS))

In [ ]:
app = Dash(__name__, external_stylesheets=[dbc.themes.CYBORG])

app.layout = dbc.Container(
    [
        dbc.Row(
            [
                dbc.Col(
                    [
                        dbc.Label("SPÉCIALITÉS", className="fw-bold"),
                        dcc.Checklist(
                            options=spe_options,
                            value=[spe_options[3]["value"]],
                            id="spe_dd",
                            inline=True,
                            inputStyle={"margin-right": "5px"},
                            labelStyle={"margin-right": "5px"}
                        ),
                        html.Br(),
                        dbc.Label("UGA", className="fw-bold"),
                        dcc.RadioItems(
                            options=uga_options,
                            value=[uga_options[i]["value"] for i in range(len(uga_options))],
                            id="uga_dd",
                            inline=True,
                            inputStyle={"margin-right": "5px"},
                            labelStyle={"margin-right": "5px"}
                        ),
                        html.Br(),
                        dbc.Label("PVM", className="fw-bold"),
                        legend1,
                        html.Br(),
                        dbc.Label("POTENTIEL", className="fw-bold"),
                        legend2
                    ],
                    width=2,
                    className="justify-content-evenly"
                ),
                dbc.Col(
                    [
                        dbc.Label("PROFESSIONNELS DE SANTÉ", className="fw-bold"),
                        dash_table.DataTable(
                            # data=id_adr.to_dict('records'),
                            columns=[{"name": i.upper(), "id": i} for i in id_adr.columns],
                            fixed_rows={'headers': True},
                            page_size=25,
                            style_table={'fontSize': 10, 'font-family': 'Verdana', 'height': '600px'},
                            style_data={"color": "black"},
                            style_data_conditional=styles,
                            style_header={'fontSize': 14, 'font_weight': 900, 'text-align': 'center',
                                          'color': 'darkblue'},
                            style_cell_conditional=[
                                {'if': {'column_id': 'id'}, 'textAlign': 'center', 'width': '45px'},
                                {'if': {'column_id': 'nom'}, 'textAlign': 'left', 'width': '220px'},
                                {'if': {'column_id': 'prenom'}, 'textAlign': 'left', 'width': '160px'},
                                {'if': {'column_id': 'spe'}, 'textAlign': 'center', 'width': '70px'},
                                {'if': {'column_id': 'pot'}, 'textAlign': 'center', 'width': '70px'},
                                {'if': {'column_id': 'pvm'}, 'textAlign': 'center', 'width': '70px'},
                                {'if': {'column_id': 'nv2022'}, 'textAlign': 'center', 'width': '70px'},
                                {'if': {'column_id': 'etablissement'}, 'textAlign': 'center', 'width': '160px',
                                 'textOverflow': 'ellipsis', 'fontSize': 10},
                                {'if': {'column_id': 'uga'}, 'textAlign': 'center', 'width': '70px', 'fontSize': 10},
                                {'if': {'column_id': 'adresse'}, 'textAlign': 'left', 'width': '230px', 'fontSize': 10},
                                {'if': {'column_id': 'cp'}, 'textAlign': 'center', 'width': '70px', 'fontSize': 10},
                                {'if': {'column_id': 'ville'}, 'textAlign': 'left', 'width': '70px', 'fontSize': 10},
                                {'if': {'column_id': 'tel'}, 'textAlign': 'center', 'width': '80px', 'fontSize': 10}
                            ],
                            css=[
                                {"selector": ".dash-spreadsheet-container tr th", "rule": "height: 16px;"},
                                # set height of header
                                {"selector": ".dash-spreadsheet-container tr", "rule": "height: 10px;"},
                                # set height of body rows
                                {"selector": ".dash-spreadsheet-inner", "rule": 'max-height: "calc(100vh - 226px)"'}
                            ],
                            id="datatable"
                        )
                    ],
                    width=9
                )
            ],
            class_name="d-flex justify-content-evenly flex-row"
        ),
        dbc.Row(

        )
    ],
    style={"margin": 20},
    fluid=True
)


@dashapp.callback(
    Output("datatable", "data"),
    Input("spe_dd", "value"),
    Input("uga_dd", "value")
)
def update_table(spes, ugas):

    id_adr = join_id_adr()

    print("selected", spes, ugas)

    if spes:
        id_adr = id_adr[id_adr["spe"].isin(spes)]
    else:
        id_adr = id_adr

    if ugas:
        if isinstance(ugas, str):
            ugas = [ugas]
        id_adr = id_adr[id_adr["uga"].isin(ugas)]
    else:
        id_adr = id_adr

    return id_adr.to_dict("records")


@app.callback(
    Output('user-store', 'data'),
    Input('my-dropdown', 'value'),
    State('user-store', 'data'))
def cur_user(args, data):
    if current_user.is_authenticated:
        return current_user.user

    
@app.callback(
    Output('username', 'children'), 
    Input('user-store', 'data')
)
def username(data):
    if data is None:
        return ''
    else:
        return f'Hello {data}'
        
app.run()org/en/14/orm/internals.html

In [ ]:
spe_options = [{"label": val, "value": val} for val in [id_adr["spe"].unique().tolist()[i] for i in [6, 3, -1, 1 ,-2 ,5 ,4 ,0 ,2]]]
uga_options = [{"label": val, "value": val} for val in id_adr["uga"].unique().tolist()]
datatable_cols =[{"name": i.upper(), "id": i} for i in id_adr.columns]

In [ ]:
spe_options = [
    {'label': 'GY', 'value': 'GY'},
    {'label': 'MG-GY', 'value': 'MG-GY'},
    {'label': 'PE-PSY', 'value': 'PE-PSY'},
    {'label': 'SF', 'value': 'SF'},
    {'label': 'PE', 'value': 'PE'},
    {'label': 'GE', 'value': 'GE'},
    {'label': 'PSY', 'value': 'PSY'},
    {'label': 'NE', 'value': 'NE'},
    {'label': 'MG', 'value': 'MG'}
]

In [ ]:
uga_options = [
    {'label': '75AUT', 'value': '75AUT'},
    {'label': '75ELY', 'value': '75ELY'},
    {'label': '75GRE', 'value': '75GRE'},
    {'label': '75INV', 'value': '75INV'},
    {'label': '75MNP', 'value': '75MNP'},
    {'label': '75PAS', 'value': '75PAS'},
    {'label': '75PER', 'value': '75PER'},
    {'label': '75TER', 'value': '75TER'},
    {'label': '75TRO', 'value': '75TRO'},
    {'label': '75VAU', 'value': '75VAU'},
    {'label': '92LEV', 'value': '92LEV'},
    {'label': '92NEU', 'value': '92NEU'}
]

In [ ]:
datatable_cols

In [ ]:
[uga_options[i]["value"] for i in range(len(uga_options))]

# Clinique Sainte Thérèse

In [ ]:
doctors = {}

for i in range(1,5):    
    r = requests.get(f"https://www.cliniquesaintetherese.fr/fr/annuaire-specialistes/page-{i}")
    soup = BeautifulSoup(r.content, "html.parser")

    cells = soup.findAll("div", class_="cell auto")
    
    for j in range(len(cells)):
        
        cell = cells[j]
        name = cell.find("a", class_="js-annuaire-detail").text
        spec_div = cell.find("div", class_="list-annuaire__list--spe").findAll("a")
        
        for k in range(len(spec_div)):
            atag = spec_div[k]
            if atag.text:
                spec = atag.text
        doctors[name] = spec

In [ ]:
therese = pd.DataFrame.from_dict(doctors, orient="index")
therese.to_excel("/home/julien/Desktop/therese.xlsx", header=None)

# GEOPY

In [ ]:
load_dotenv(verbose=True)

In [ ]:
SQLALCHEMY_DATABASE_URI = environ.get("SQLALCHEMY_DATABASE_URI")

In [ ]:
SQLALCHEMY_DATABASE_URI

In [ ]:
import sqlite3
sqliteConnection = sqlite3.connect("app/data/biocodex.db")
cursor = sqliteConnection.cursor()

In [ ]:
from app import create_flask_server, dash_apps_factory

In [ ]:
from app.dashboards.biocodex.models import Adress

In [ ]:
from app.extensions import db

In [ ]:
engine = create_engine(
    'sqlite:////home/julien/website/app/data/biocodex.db'
)

with Session(engine) as session:
    id_cdb = pd.read_sql_query(            
        sql = session.query(
            Connections.doc_id, Identity.nom, Identity.prenom, Identity.spe, Identity.pot, Identity.pvm, Identity.nv2022,
            Adress.adresse, Adress.cp, Adress.ville, Adress.uga, Adress.tel).join(Identity).join(Adress).filter(Adress.id==1).statement,
        con=engine
    )
    session.close()

In [ ]:
from os import environ
from dotenv import load_dotenv

In [ ]:
engine = create_engine(
    'sqlite:////home/julien/website/app/data/biocodex.db'
)

with Session(engine) as session:
    adresses = pd.read_sql_query(
        session.query(Adress.id, Adress.adresse, Adress.cp, Adress.ville).statement,
        con=engine
    )

In [ ]:
from tqdm import notebook

In [ ]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="mySecretAgent")

sqliteConnection = sqlite3.connect("app/data/biocodex.db")
cursor = sqliteConnection.cursor()

for i, row in adresses.iterrows():
    adr = row["adresse"] + " " + row["cp"] + " " +row["ville"]
    location = geolocator.geocode(adr)
    id = row['id']
    cursor.execute(f"UPDATE adresses SET lat=?, lon=? WHERE id=?", (location.latitude, location.longitude, id))
    
    

In [ ]:
from sqlalchemy import text

In [ ]:
import sqlite3


In [ ]:
query = """
BEGIN TRANSACTION;
ALTER TABLE adresses ADD lat REAL DEFAULT 0;
ALTER TABLE adresses ADD mat REAL DEFAULT 0;
COMMIT
"""

In [ ]:
# write the SQL query inside the text() block
sql_query = text(query)

In [ ]:
cursor.execute("ALTER TABLE adresses ADD lon REAL DEFAULT 0;")

In [ ]:
query="SELECT id, adresse, cp ,ville FROM adresses"
results = cursor.execute(query)

In [ ]:
# View the records
for record in results:
    print("\n", record)

In [ ]:
query = """
BEGIN TRANSACTION;
ALTER TABLE adresses ADD lat REAL DEFAULT 0;
ALTER TABLE adresses ADD mat REAL DEFAULT 0;
COMMIT
"""

In [ ]:
result = engine.execute(query).fetchall()
 
# print all the records
for i in result:
    print("\n", i)